# 策略研究和回测笔记

用于开发和测试量化交易策略

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.data_collector.data_manager import data_manager
from src.strategy.technical import TechnicalStrategy, MACDStrategy, MaCrossoverStrategy
from src.strategy.multi_factor import SimpleMultiFactorStrategy
from src.backtest.engine import BacktestEngine
from src.backtest.performance import PerformanceAnalyzer
from src.utils.database import init_db

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
# 初始化
init_db()

## 1. 数据准备

In [ ]:
# 定义股票池
stock_pool = ['600000.SH', '600036.SH', '600519.SH', '000001.SZ', '000651.SZ',
              '000858.SZ', '600276.SH', '601318.SH', '601888.SH', '603259.SH']

# 更新数据
for ts_code in stock_pool:
    data_manager.update_single_stock(ts_code, days=365)
    print(f"已更新 {ts_code}")

In [ ]:
# 加载数据
end_date = datetime.now().strftime('%Y%m%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y%m%d')

market_data = {}
for ts_code in stock_pool:
    df = data_manager.get_daily_quotes(ts_code, start_date, end_date)
    if not df.empty:
        df['trade_date'] = pd.to_datetime(df['trade_date'])
        market_data[ts_code] = df

print(f"加载了 {len(market_data)} 只股票的数据")

## 2. 技术指标策略回测

In [ ]:
# 创建均线交叉策略
strategy = MaCrossoverStrategy(
    name="ma_crossover_5_20",
    short_period=5,
    long_period=20
)

print(f"策略：{strategy.name}")
print(f"参数：短期均线={strategy.short_period}, 长期均线={strategy.long_period}")

In [ ]:
# 创建回测引擎
engine = BacktestEngine(
    initial_capital=1000000,  # 100 万
    commission_rate=0.0003,   # 万分之三
    stamp_tax_rate=0.001,     # 千分之一
    slippage_rate=0.001       # 千分之一滑点
)

# 设置策略
engine.set_strategy(strategy)

# 加载数据
data_dict = engine.load_data(
    ts_codes=list(market_data.keys()),
    start_date=start_date,
    end_date=end_date
)

In [ ]:
# 运行回测
result = engine.run(data_dict)

In [ ]:
# 绩效分析
analyzer = PerformanceAnalyzer()
report = analyzer.generate_report(result)
print(report)

In [ ]:
# 绘制权益曲线
if not result.equity_curve.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # 权益曲线
    axes[0].plot(result.equity_curve.index, result.equity_curve['total_equity'])
    axes[0].set_title('权益曲线')
    axes[0].set_xlabel('交易日')
    axes[0].set_ylabel('总权益 (元)')
    axes[0].grid(True, alpha=0.3)
    
    # 回撤曲线
    equity = result.equity_curve['total_equity']
    peak = equity.expanding(min_periods=1).max()
    drawdown = (equity - peak) / peak * 100
    
    axes[1].fill_between(range(len(drawdown)), drawdown, 0, color='red', alpha=0.5)
    axes[1].set_title('回撤 (%)')
    axes[1].set_xlabel('交易日')
    axes[1].set_ylabel('回撤 %')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 3. 参数优化

In [ ]:
# 测试不同的均线参数组合
param_test_results = []

for short_period in [3, 5, 10]:
    for long_period in [20, 30, 60]:
        # 创建策略
        test_strategy = MaCrossoverStrategy(
            name=f"ma_{short_period}_{long_period}",
            short_period=short_period,
            long_period=long_period
        )
        
        # 创建引擎
        test_engine = BacktestEngine(initial_capital=1000000)
        test_engine.set_strategy(test_strategy)
        test_engine.load_data(
            ts_codes=list(market_data.keys()),
            start_date=start_date,
            end_date=end_date
        )
        
        # 运行回测
        test_result = test_engine.run(data_dict)
        
        param_test_results.append({
            'short_period': short_period,
            'long_period': long_period,
            'total_return': test_result.total_return,
            'sharpe_ratio': test_result.sharpe_ratio,
            'max_drawdown': test_result.max_drawdown,
            'total_trades': test_result.total_trades
        })

# 转换为 DataFrame
results_df = pd.DataFrame(param_test_results)
print("\n参数优化结果:")
print(results_df.sort_values('sharpe_ratio', ascending=False))

In [ ]:
# 可视化参数优化结果
pivot_data = results_df.pivot(index='short_period', columns='long_period', values='sharpe_ratio')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('不同均线参数的夏普比率')
plt.xlabel('长期均线')
plt.ylabel('短期均线')
plt.tight_layout()
plt.show()

## 4. 多因子策略测试

In [ ]:
# 创建简单多因子策略
factor_strategy = SimpleMultiFactorStrategy(
    name="simple_multi_factor",
    factors=['pe', 'roe', 'momentum'],
    top_n=3,
    rebalance_days=10
)

# 准备因子数据
factor_data = []
for ts_code in stock_pool:
    financial_df = data_manager.get_financial_indicators(ts_code)
    if not financial_df.empty:
        row = financial_df.iloc[0]
        factor_data.append({
            'ts_code': ts_code,
            'pe': row.get('pe', 0),
            'pb': row.get('pb', 0),
            'roe': row.get('roe', 0),
            'momentum': 0  # 可以计算动量
        })

if factor_data:
    factor_df = pd.DataFrame(factor_data)
    factor_strategy.set_factor_data(factor_df)
    print("因子数据:")
    print(factor_df)

## 5. 总结

- 测试了均线交叉策略
- 进行了参数优化
- 测试了多因子策略

### 下一步
1. 增加更多策略进行测试
2. 优化参数
3. 进行更详细的归因分析